In [0]:
%pip install -q "psycopg[binary]>=3.1.0"

# Lakebase FGAC Validation
**Database:** fe_bar_db | **Project:** fe-bar-fraud-analytics | **Branch:** production

Validates the fine-grained access control layer created by `lakebase_fgac.sql`:
- Governed views in `fraud_gov` schema (security_barrier, row + column rules)
- RLS policies on `cases.fraud_cases` and `cases.case_actions`
- Column-level GRANTs on fraud_cases for fraud_analyst
- Three role personas: fraud_analyst (non-pii), fraud_lead (pii), fraud_triage_agent (pii)

In [0]:
import psycopg, time, sys
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
username = w.current_user.me().user_name
ep = w.postgres.get_endpoint(name="projects/fe-bar-fraud-analytics/branches/production/endpoints/primary")
host = ep.status.hosts.host
token = w.postgres.generate_database_credential(endpoint=ep.name).token

def get_conn():
    return psycopg.connect(host=host, dbname="fe_bar_db", user=username, password=token, sslmode="require")

def run_as_role(role, sql, params=None, fetch=True):
    """Execute SQL as the given Postgres role via SET ROLE."""
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute(psycopg.sql.SQL("SET ROLE {}").format(psycopg.sql.Identifier(role)))
            cur.execute(sql, params)
            if fetch and cur.description:
                cols = [d.name for d in cur.description]
                rows = cur.fetchall()
                return cols, rows
            conn.commit()
            return None, None
    finally:
        conn.close()

def expect_error(role, sql, params=None, label=""):
    """Run SQL as role and expect a permission/policy error. Returns error text."""
    try:
        run_as_role(role, sql, params)
        return f"UNEXPECTED SUCCESS ({label})"
    except Exception as e:
        return str(e).split('\n')[0]

print(f"Connected as {username} to {host}")

In [0]:
print("=" * 80)
print("B.5  VALIDATION MATRIX: fraud_gov views")
print("=" * 80)

roles = ['fraud_analyst', 'fraud_lead', 'fraud_triage_agent']
results = []

for role in roles:
    # Total rows in fraud_gov.txn_features
    _, rows = run_as_role(role, "SELECT COUNT(*) FROM fraud_gov.txn_features")
    total = rows[0][0]
    
    # Rows with amount > 10000
    _, rows = run_as_role(role, "SELECT COUNT(*) FROM fraud_gov.txn_features WHERE amount > 10000")
    high_amt = rows[0][0]
    
    # Non-null amt_to_monthly_spend
    _, rows = run_as_role(role, "SELECT COUNT(amt_to_monthly_spend) FROM fraud_gov.txn_features")
    non_null_amt = rows[0][0]
    
    # Cases visible
    _, rows = run_as_role(role, "SELECT COUNT(*) FROM cases.fraud_cases")
    cases_visible = rows[0][0]
    
    results.append({
        'role': role,
        'txn_features_rows': total,
        'high_amount_rows': high_amt,
        'non_null_amt_spend': non_null_amt,
        'cases_visible': cases_visible,
    })
    print(f"\n{role}:")
    print(f"  txn_features rows:     {total}")
    print(f"  amount > 10000 rows:   {high_amt}")
    print(f"  non-null amt_to_spend: {non_null_amt}")
    print(f"  cases visible:         {cases_visible}")

print("\n" + "=" * 80)
print("Expected: analyst sees 463,881 rows (no high-amt), 0 non-null amt_to_spend")
print("Expected: lead and agent see 464,632 rows (incl high-amt), ~431K non-null")
print("=" * 80)

================================================================================B.5  VALIDATION MATRIX: fraud_gov views================================================================================fraud_analyst:  txn_features rows:     463881  amount > 10000 rows:   0  non-null amt_to_spend: 0  cases visible:         100fraud_lead:  txn_features rows:     464632  amount > 10000 rows:   751  non-null amt_to_spend: 431471  cases visible:         104fraud_triage_agent:  txn_features rows:     464632  amount > 10000 rows:   751  non-null amt_to_spend: 431471  cases visible:         104

In [0]:
print("=" * 80)
print("B.5  EXPECTED-DENY TESTS")
print("=" * 80)

deny_tests = [
    # Analyst INSERT a case
    ('fraud_analyst', 
     "INSERT INTO cases.fraud_cases (txn_id, recommended_action, assigned_queue, created_by) VALUES ('TXN-DENY-TEST', 'review', 'analyst_review', 'test')",
     None, 'analyst INSERT case'),
    # Analyst UPDATE notes
    ('fraud_analyst',
     "UPDATE cases.fraud_cases SET notes = 'hacked' WHERE case_id = gen_random_uuid()",
     None, 'analyst UPDATE notes'),
    # Agent UPDATE a case
    ('fraud_triage_agent',
     "UPDATE cases.fraud_cases SET status = 'closed' WHERE case_id = gen_random_uuid()",
     None, 'agent UPDATE case'),
    # Agent DELETE a case
    ('fraud_triage_agent',
     "DELETE FROM cases.fraud_cases WHERE case_id = gen_random_uuid()",
     None, 'agent DELETE case'),
    # Agent INSERT with wrong created_by
    ('fraud_triage_agent',
     "INSERT INTO cases.fraud_cases (txn_id, recommended_action, assigned_queue, created_by) VALUES ('TXN-0000001', 'review', 'analyst_review', 'someone_else')",
     None, 'agent INSERT wrong created_by'),
    # Any role SELECT on fraud.txn_features directly
    ('fraud_analyst',
     "SELECT COUNT(*) FROM fraud.txn_features",
     None, 'analyst direct SELECT fraud.txn_features'),
    ('fraud_lead',
     "SELECT COUNT(*) FROM fraud.txn_features",
     None, 'lead direct SELECT fraud.txn_features'),
    ('fraud_triage_agent',
     "SELECT COUNT(*) FROM fraud.txn_features",
     None, 'agent direct SELECT fraud.txn_features'),
]

for role, sql, params, label in deny_tests:
    err = expect_error(role, sql, params, label)
    status = 'DENIED' if ('permission denied' in err.lower() or 'violates row-level' in err.lower() or 'new row violates' in err.lower()) else 'UNEXPECTED'
    print(f"  [{status}] {label}: {err[:120]}")

================================================================================B.5  EXPECTED-DENY TESTS================================================================================  [DENIED] analyst INSERT case: permission denied for table fraud_cases  [DENIED] analyst UPDATE notes: permission denied for table fraud_cases  [DENIED] agent UPDATE case: permission denied for table fraud_cases  [DENIED] agent DELETE case: permission denied for table fraud_cases  [DENIED] agent INSERT wrong created_by: new row violates row-level security policy for table "fraud_cases"  [DENIED] analyst direct SELECT fraud.txn_features: permission denied for table txn_features  [DENIED] lead direct SELECT fraud.txn_features: permission denied for table txn_features  [DENIED] agent direct SELECT fraud.txn_features: permission denied for table txn_features

In [0]:
import sys
sys.path.insert(0, "/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo")
if "lakebase_cases" in sys.modules:
    del sys.modules["lakebase_cases"]
from lakebase_cases import LakebaseCases

print("=" * 80)
print("B.5  EXPECTED-ALLOW TESTS")
print("=" * 80)

# First create a test case as fraud_triage_agent for analyst to work with
# Use a non-traced txn that is analyst_review and amount <= 10000
conn = get_conn()
with conn.cursor() as cur:
    cur.execute("SET ROLE NONE")  # use superuser role
    # Find a review-band txn with amount <= 10000
    cur.execute("""
        SELECT tr.txn_id FROM fraud.txn_risk_current tr
        JOIN fraud.txn_features tf ON tr.txn_id = tf.txn_id
        WHERE tr.recommended_action = 'review' AND tf.amount <= 10000
        AND tr.txn_id <> 'TXN-0234209'
        LIMIT 1
    """)
    test_txn = cur.fetchone()[0]
    print(f"Test txn for analyst allow tests: {test_txn}")
conn.close()

# Test 1: Agent INSERT a case via LakebaseCases
print("\n--- Agent INSERT via LakebaseCases ---")
lc_agent = LakebaseCases(pg_role='fraud_triage_agent')
try:
    test_case_id = lc_agent.create_fraud_case(
        txn_id=test_txn,
        recommended_action='review',
        assigned_queue='analyst_review',
        notes='FGAC allow test',
        created_by='fraud_triage_copilot:test@databricks.com'
    )
    print(f"  [ALLOWED] Agent created case {test_case_id} for {test_txn}")
except Exception as e:
    print(f"  [FAILED] Agent INSERT: {e}")
    test_case_id = None
finally:
    lc_agent.close()

# Test 2: Analyst UPDATE status on visible analyst_review case
if test_case_id:
    print("\n--- Analyst UPDATE status ---")
    try:
        conn = get_conn()
        with conn.cursor() as cur:
            cur.execute(psycopg.sql.SQL("SET ROLE {}").format(psycopg.sql.Identifier('fraud_analyst')))
            cur.execute(
                "UPDATE cases.fraud_cases SET status = 'in_review', updated_at = now() WHERE case_id = %s",
                (test_case_id,)
            )
            print(f"  [ALLOWED] Analyst updated status to in_review on case {test_case_id}")
        conn.commit()
        conn.close()
    except Exception as e:
        print(f"  [FAILED] Analyst UPDATE: {e}")

# Clean up: close the test case as superuser
if test_case_id:
    print("\n--- Cleanup: closing test case as superuser ---")
    lc_su = LakebaseCases()
    lc_su.update_case_status(test_case_id, 'closed', 'fgac_test_cleanup')
    print(f"  Closed case {test_case_id}")
    lc_su.close()

print("\nAll allow tests passed.")

================================================================================B.5  EXPECTED-ALLOW TESTS================================================================================Test txn for allow tests: TXN-0124326--- Agent INSERT via LakebaseCases(pg_role='fraud_triage_agent') ---  [ALLOWED] Agent created case 2153e489-8e24-434d-9631-4f7aa215c4da for TXN-0124326--- Analyst UPDATE status ---  [ALLOWED] Analyst updated status to in_review on case 2153e489-8e24-434d-9631-4f7aa215c4da--- Cleanup ---  Closed case 2153e489-8e24-434d-9631-4f7aa215c4daAll allow tests passed.

In [0]:
import random, statistics

print("=" * 80)
print("B.5  READ LATENCY: 200 PK lookups on fraud_gov.txn_risk_current")
print("=" * 80)

# Get 200 random txn_ids
conn = get_conn()
with conn.cursor() as cur:
    cur.execute("SELECT txn_id FROM fraud.txn_risk_current ORDER BY random() LIMIT 200")
    txn_ids = [r[0] for r in cur.fetchall()]
conn.close()

# Run lookups as fraud_triage_agent
conn = get_conn()
latencies = []
with conn.cursor() as cur:
    cur.execute(psycopg.sql.SQL("SET ROLE {}").format(psycopg.sql.Identifier('fraud_triage_agent')))
    for txn_id in txn_ids:
        t0 = time.time()
        cur.execute("SELECT * FROM fraud_gov.txn_risk_current WHERE txn_id = %s", (txn_id,))
        cur.fetchone()
        latencies.append((time.time() - t0) * 1000)  # ms
conn.close()

latencies.sort()
p50 = latencies[len(latencies) // 2]
p95 = latencies[int(len(latencies) * 0.95)]
p99 = latencies[int(len(latencies) * 0.99)]

print(f"  Lookups: {len(latencies)}")
print(f"  p50: {p50:.2f} ms")
print(f"  p95: {p95:.2f} ms")
print(f"  p99: {p99:.2f} ms")
print(f"  min: {min(latencies):.2f} ms, max: {max(latencies):.2f} ms")

================================================================================B.5  READ LATENCY: 200 PK lookups on fraud_gov.txn_risk_current as fraud_triage_agent================================================================================  Lookups: 200  p50: 3.82 ms  p95: 5.81 ms  p99: 7.98 ms  min: 2.43 ms, max: 8.05 ms

In [0]:
print("=" * 80)
print("B.6  UC PATH GOVERNANCE REPORT")
print("=" * 80)

# Query fraud_gov views through UC (via Spark SQL)
result = spark.sql("""
    SELECT 
        'uc_fraud_gov_total' as check_type, COUNT(*) as cnt 
    FROM fe_bar_lakebase.fraud_gov.txn_features
    UNION ALL
    SELECT 'uc_fraud_gov_high_amt', COUNT(*) 
    FROM fe_bar_lakebase.fraud_gov.txn_features WHERE amount > 10000
    UNION ALL
    SELECT 'uc_fraud_gov_non_null_amt', COUNT(amt_to_monthly_spend) 
    FROM fe_bar_lakebase.fraud_gov.txn_features
    UNION ALL
    SELECT 'uc_fraud_direct_total', COUNT(*) 
    FROM fe_bar_lakebase.fraud.txn_features
    UNION ALL
    SELECT 'uc_fraud_direct_high_amt', COUNT(*) 
    FROM fe_bar_lakebase.fraud.txn_features WHERE amount > 10000
    UNION ALL
    SELECT 'uc_fraud_direct_non_null_amt', COUNT(amt_to_monthly_spend) 
    FROM fe_bar_lakebase.fraud.txn_features
""").collect()

for row in result:
    print(f"  {row['check_type']}: {row['cnt']}")

print("\nFINDINGS:")
print("  1. fraud_gov views ARE visible through UC (fe_bar_lakebase.fraud_gov.*)")
print("  2. pg_has_role() always returns FALSE through UC path (UC uses a system")
print("     Postgres role, not the user's personal role). Therefore:")
print("     - Row filter: high-amount rows always hidden through fraud_gov via UC")
print("     - Column mask: amt_to_monthly_spend always NULL through fraud_gov via UC")
print("  3. Direct fraud.* synced tables through UC have NO Postgres FGAC")
print("     (UC FGAC on the silver tables does NOT propagate to synced tables)")
print("  4. Effective FGAC enforcement requires direct Postgres with SET ROLE")

================================================================================B.6  UC PATH GOVERNANCE REPORT================================================================================  uc_fraud_gov_total: 463881  uc_fraud_gov_high_amt: 0  uc_fraud_gov_non_null_amt: 0  uc_fraud_direct_total: 464632  uc_fraud_direct_high_amt: 751  uc_fraud_direct_non_null_amt: 431471FINDINGS:  1. fraud_gov views ARE visible through UC (fe_bar_lakebase.fraud_gov.*)  2. pg_has_role() always returns FALSE through UC path (UC uses a system     Postgres role, not the user's personal role). Therefore:     - Row filter: high-amount rows always hidden through fraud_gov via UC     - Column mask: amt_to_monthly_spend always NULL through fraud_gov via UC  3. Direct fraud.* synced tables through UC have NO Postgres FGAC  4. Effective FGAC enforcement requires direct Postgres with SET ROLE